# 相手の未公開情報を推定する

TreeSearchPlayer.estimate_opponent() をオーバーライドして、相手ポケモンの見えていない技を推定値で補完する。

In [1]:
%pip install -q jpoke

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: C:\Users\tmtmh\Documents\pokemon\jpoke\.venv\Scripts\python.exe -m pip install --upgrade pip


In [2]:
from jpoke import Battle, Player
from jpoke.players import RandomPlayer, TreeSearchPlayer

In [3]:
class OpponentGuessingPlayer(TreeSearchPlayer):
    """相手の未公開の技を「みずでっぽう」1本と仮定して探索するAI（estimate_opponent()の拡張例）。"""

    def estimate_opponent(self, battle: Battle, opponent: Player) -> None:
        # 実戦では対戦データベースや使用率統計等から推定するが、ここでは固定の
        # 推測技で最小例を示す。opponent.active.moves が空（未公開）のときだけ
        # 書き込む
        opponent_active = battle.get_active(opponent)
        if not opponent_active.moves:
            opponent_active.set_moves(["ハイドロポンプ"])

In [4]:
ai_player = OpponentGuessingPlayer("GuessingAI", max_plies=1, max_nodes=50)
ai_player.add_pokemon("リザードン", moves=["かえんほうしゃ", "じしん"])

opponent_player = RandomPlayer("Opponent")
opponent_player.add_pokemon("カメックス", moves=["ハイドロポンプ"])

battle = Battle(ai_player, opponent_player, seed=1)
battle.start()

# 1ターン目: 相手の技は未公開。estimate_opponent が推定を書き込むため
# fallback() には委譲されず、推定した相手の技も踏まえた探索が行われる
# （nodes_expanded > 0 になる。何も推定しない既定実装のままだと0のまま）
battle.step()
print(f"1ターン目に展開したノード数: {ai_player.nodes_expanded}")
battle.print_logs()

1ターン目に展開したノード数: 8
Turn 1 : GuessingAI : リザードン : テラスタル化した（タイプ: ほのお）
Turn 1 : Opponent : カメックス : テラスタル化した（タイプ: みず）
Turn 1 : GuessingAI : リザードン : じしん PP -1
Turn 1 : Opponent : カメックス : HP -38 (116/154)
Turn 1 : Opponent : カメックス : ハイドロポンプ PP -1
Turn 1 : GuessingAI : リザードン : HP -153 (0/153)
Turn 1 : Opponent :  : 勝利
Turn 1 : GuessingAI :  : 敗北
